# 03 Hyperparameter Search with MLflow

This notebook builds on the feature engineering notebook.

The objective is to run a structured hyperparameter search and track the search process using MLflow.

This notebook covers:

1. Loading engineered features
2. Defining a repeatable feature engineering function
3. Running GridSearchCV
4. Logging the parent search run
5. Logging individual parameter trials as MLflow child runs
6. Saving the best tuned model candidate

This is demo/portfolio code only. It does not use real patient data and must not be used for medical decision-making.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import json

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_PATH)
print("Package exists:", (SRC_PATH / "pharma_trial_ml").exists())

In [ ]:
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pharma_trial_ml.config import (
    DATA_PATH,
    DATA_DIR,
    MODEL_DIR,
    EXPERIMENT_NAME,
    MLFLOW_TRACKING_URI,
)
from pharma_trial_ml.data import generate_mock_trial_data
from pharma_trial_ml.features import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    TARGET,
)

## 2. Load data

In [ ]:
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    df = generate_mock_trial_data(n_rows=2500, random_state=42)
    df.to_csv(DATA_PATH, index=False)

df.head()

## 3. Recreate engineered features

In [ ]:
def engineer_features(input_df: pd.DataFrame) -> pd.DataFrame:
    df_fe = input_df.copy()

    df_fe["dose_adherence_exposure"] = df_fe["dose_mg"] * (df_fe["adherence_pct"] / 100)
    df_fe["biomarker_per_dose"] = df_fe["baseline_biomarker"] / (df_fe["dose_mg"] + 1)
    df_fe["high_adherence"] = (df_fe["adherence_pct"] >= 90).astype(int)
    df_fe["older_patient"] = (df_fe["age"] >= 65).astype(int)
    df_fe["high_bmi"] = (df_fe["bmi"] >= 30).astype(int)
    df_fe["multiple_prior_treatments"] = (df_fe["prior_treatments"] >= 2).astype(int)

    df_fe["active_high_adherence"] = (
        (df_fe["trial_arm"] == "active") & (df_fe["adherence_pct"] >= 90)
    ).astype(int)

    df_fe["age_group"] = pd.cut(
        df_fe["age"],
        bins=[17, 40, 55, 65, 100],
        labels=["18_40", "41_55", "56_65", "66_plus"],
    ).astype(str)

    df_fe["bmi_group"] = pd.cut(
        df_fe["bmi"],
        bins=[0, 18.5, 25, 30, 100],
        labels=["underweight", "normal", "overweight", "obese"],
    ).astype(str)

    df_fe["biomarker_group"] = pd.cut(
        df_fe["baseline_biomarker"],
        bins=[0, 40, 60, 80, 200],
        labels=["low", "medium", "high", "very_high"],
    ).astype(str)

    df_fe["adherence_group"] = pd.cut(
        df_fe["adherence_pct"],
        bins=[0, 70, 85, 95, 100],
        labels=["low", "medium", "high", "very_high"],
        include_lowest=True,
    ).astype(str)

    return df_fe


df_fe = engineer_features(df)
df_fe.head()

## 4. Define features and preprocessor

In [ ]:
engineered_numeric_features = NUMERIC_FEATURES + [
    "dose_adherence_exposure",
    "biomarker_per_dose",
    "high_adherence",
    "older_patient",
    "high_bmi",
    "multiple_prior_treatments",
    "active_high_adherence",
]

engineered_categorical_features = CATEGORICAL_FEATURES + [
    "age_group",
    "bmi_group",
    "biomarker_group",
    "adherence_group",
]

engineered_feature_columns = engineered_numeric_features + engineered_categorical_features

X = df_fe[engineered_feature_columns]
y = df_fe[TARGET]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), engineered_numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), engineered_categorical_features),
    ]
)

X.head()

## 5. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

## 6. Define search spaces

This notebook uses `GridSearchCV` because it is easy to understand and good for a portfolio demonstration.

In real production work, `Optuna` or another Bayesian optimisation method is often more efficient than a full grid search.

In [ ]:
search_configs = {
    "logistic_regression": {
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                    ),
                ),
            ]
        ),
        "param_grid": {
            "classifier__C": [0.1, 0.5, 1.0, 3.0],
            "classifier__penalty": ["l2"],
        },
    },
    "random_forest": {
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    RandomForestClassifier(
                        random_state=42,
                        class_weight="balanced",
                    ),
                ),
            ]
        ),
        "param_grid": {
            "classifier__n_estimators": [100, 200, 300],
            "classifier__max_depth": [4, 6, 8],
            "classifier__min_samples_leaf": [4, 8, 12],
        },
    },
    "gradient_boosting": {
        "pipeline": Pipeline(
            steps=[
                ("preprocessor", preprocessor),
                (
                    "classifier",
                    GradientBoostingClassifier(
                        random_state=42,
                    ),
                ),
            ]
        ),
        "param_grid": {
            "classifier__n_estimators": [100, 200],
            "classifier__learning_rate": [0.03, 0.05, 0.1],
            "classifier__max_depth": [2, 3],
        },
    },
}

## 7. Helper functions

In [ ]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "roc_auc": roc_auc_score(y_test, probabilities),
    }


def clean_param_name(param_name: str) -> str:
    return param_name.replace("classifier__", "")

## 8. Run GridSearchCV and log to MLflow

This uses a parent run for each model family and child runs for each individual grid-search trial.

That structure makes MLflow easier to read:

```text
parent run: random_forest_search
    child run: parameter combination 1
    child run: parameter combination 2
    child run: parameter combination 3
```

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_search_results = []
best_overall_model = None
best_overall_details = None
best_overall_score = -1

for model_name, config in search_configs.items():
    with mlflow.start_run(run_name=f"03_{model_name}_grid_search") as parent_run:
        mlflow.log_param("notebook", "03_hyperparameter_search_mlflow")
        mlflow.log_param("model_family", model_name)
        mlflow.log_param("cv_folds", 5)
        mlflow.log_param("scoring", "f1")

        grid_search = GridSearchCV(
            estimator=config["pipeline"],
            param_grid=config["param_grid"],
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            return_train_score=True,
        )

        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        test_metrics = evaluate_model(best_model, X_test, y_test)

        mlflow.log_params({f"best_{clean_param_name(k)}": v for k, v in grid_search.best_params_.items()})
        mlflow.log_metric("best_cv_f1", grid_search.best_score_)
        mlflow.log_metrics({f"test_{k}": v for k, v in test_metrics.items()})

        # Log each parameter combination as a child run
        cv_results = pd.DataFrame(grid_search.cv_results_)

        for _, row in cv_results.iterrows():
            with mlflow.start_run(
                run_name=f"03_{model_name}_trial",
                nested=True,
            ):
                params = row["params"]
                mlflow.log_param("parent_model_family", model_name)
                mlflow.log_params({clean_param_name(k): v for k, v in params.items()})

                mlflow.log_metric("mean_cv_f1", row["mean_test_score"])
                mlflow.log_metric("std_cv_f1", row["std_test_score"])
                mlflow.log_metric("mean_train_f1", row["mean_train_score"])
                mlflow.log_metric("rank_test_score", int(row["rank_test_score"]))

        # Save plots for best model
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_estimator(best_model, X_test, y_test, ax=ax)
        ax.set_title(f"Best {model_name}: confusion matrix")
        confusion_path = MODEL_DIR / f"03_best_{model_name}_confusion_matrix.png"
        fig.savefig(confusion_path, bbox_inches="tight")
        plt.close(fig)
        mlflow.log_artifact(str(confusion_path), artifact_path="plots")

        fig, ax = plt.subplots()
        RocCurveDisplay.from_estimator(best_model, X_test, y_test, ax=ax)
        ax.set_title(f"Best {model_name}: ROC curve")
        roc_path = MODEL_DIR / f"03_best_{model_name}_roc_curve.png"
        fig.savefig(roc_path, bbox_inches="tight")
        plt.close(fig)
        mlflow.log_artifact(str(roc_path), artifact_path="plots")

        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="best_model",
            input_example=X_test.head(3),
        )

        details = {
            "model_family": model_name,
            "parent_run_id": parent_run.info.run_id,
            "best_cv_f1": grid_search.best_score_,
            "best_params": grid_search.best_params_,
            **{f"test_{k}": v for k, v in test_metrics.items()},
        }

        all_search_results.append(details)

        if test_metrics["f1"] > best_overall_score:
            best_overall_score = test_metrics["f1"]
            best_overall_model = best_model
            best_overall_details = details

search_results_df = pd.DataFrame(all_search_results).sort_values("test_f1", ascending=False)
search_results_df

## 9. Save the best tuned model candidate

In [ ]:
TUNED_MODEL_PATH = MODEL_DIR / "trial_response_tuned_model.joblib"
TUNED_METADATA_PATH = MODEL_DIR / "tuned_model_metadata.json"

joblib.dump(best_overall_model, TUNED_MODEL_PATH)

metadata = {
    "model_path": str(TUNED_MODEL_PATH),
    "best_overall_details": best_overall_details,
    "engineered_features": engineered_feature_columns,
    "target": TARGET,
    "selection_metric": "test_f1",
    "notes": [
        "GridSearchCV used with 5-fold stratified cross-validation.",
        "Each parameter combination logged as a nested MLflow child run.",
        "Best final model selected using held-out test F1.",
        "This is portfolio/demo code only.",
    ],
}

TUNED_METADATA_PATH.write_text(json.dumps(metadata, indent=2))

metadata

## 10. Test inference with tuned model

In [ ]:
loaded_model = joblib.load(TUNED_MODEL_PATH)

sample_raw = pd.DataFrame(
    [
        {
            "age": 54,
            "bmi": 29.5,
            "baseline_biomarker": 68.2,
            "dose_mg": 50,
            "adherence_pct": 92,
            "prior_treatments": 1,
            "sex": "F",
            "trial_arm": "active",
        }
    ]
)

sample_fe = engineer_features(sample_raw)

probability = loaded_model.predict_proba(sample_fe[engineered_feature_columns])[0, 1]
prediction = int(probability >= 0.5)

{
    "prediction": prediction,
    "responder_probability": round(float(probability), 4),
}

## 11. How this fits production

At this point the project has:

```text
01_initial_experiment.ipynb
    Simple baseline models

02_feature_engineering.ipynb
    Feature engineering and raw-vs-engineered comparison

03_hyperparameter_search_mlflow.ipynb
    Grid search, nested MLflow runs, tuned model candidate
```

The next production step would be to move the feature engineering function into:

```text
src/pharma_trial_ml/feature_engineering.py
```

Then the API should use that same function before making predictions. This prevents training-serving skew.